# Ontario Fire Disturbance Point: cleaning

**Project:** Wind-Based Wildfire Ignition and Spread Prediction System for Ontario (COMP 385 AI Capstone)
**Dataset owner on the team:** Rishi Goyal
**Builds on:** `01_fire_disturbance_point_eda.ipynb`, which found the problems this notebook fixes.

**Ground rule:** the raw download `Fire_Disturbance_Point.csv` is never modified. Each cleaning step works on a copy, and the result is saved under `data/interim/`.

**Cleaning steps**
1. Remove columns we don't need *(this version)*
2. Remove rows that can't be ignition labels *(next)*
3. Parse dates and fix known date errors *(next)*
4. Decode codes and add flag columns *(next)*

## Step 1. Load the raw file

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# Works whether the notebook runs from the project root or from notebooks/
ROOT = next(p for p in [Path("."), Path("..")] if (p / "Fire_Disturbance_Point.csv").exists())
RAW_PATH = ROOT / "Fire_Disturbance_Point.csv"
OUT_DIR = ROOT / "data" / "interim"

raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig", low_memory=False)
print(f"Raw file: {RAW_PATH.resolve()}")
print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")

Raw file: C:\Users\rishi\OneDrive\Desktop\Ai capstone\Fire_Disturbance_Point.csv
Rows: 65,655   Columns: 17


## Step 2. Decide which columns to keep

Each column is judged against what the project needs: ignition labels for the classifier, final size and dates for the spread checks, and a key to join points to the Area layer.

| Column | Decision | Reason |
|---|---|---|
| `OGF_ID` | Keep | Unique record key on every row |
| `X`, `Y` | Keep | Ignition location, NAD83 longitude and latitude |
| `FIRE_DISTURBANCE_AREA_IDENT` | Keep | With year and type, joins points to Area polygons |
| `FIRE_TYPE_CODE` | Keep | Needed to remove prescribed burns and to join to the Area layer |
| `FIRE_YEAR` | Keep | Part of the join key; shows over-winter holdovers |
| `FIRE_GENERAL_CAUSE_CODE` | Keep | Results must be reported by cause |
| `FIRE_START_DATE` | Keep | The ignition label date |
| `FIRE_OUT_DATE` | Keep | Fire duration for spread checks |
| `FIRE_FINAL_SIZE` | Keep | Size classes and matching to Area polygons |
| `FIRE_WEATHER_INDEX` | **Drop** | Exists only on fire rows, so it leaks the label. The raw file keeps it for the FWI cross-check. |
| `FIRE_RESPONSE_OBJ_CODE` | **Drop** | Decided after ignition, so it leaks the label; missing before 1990 |
| `OBJECTID` | **Drop** | Export row number; repeats what `OGF_ID` does |
| `BUSINESS_EFFECTIVE_DATE` | **Drop** | Database bookkeeping; 74% missing |
| `GEOMETRY_UPDATE_DATETIME` | **Drop** | Database bookkeeping |
| `EFFECTIVE_DATETIME` | **Drop** | Database bookkeeping |
| `SYSTEM_DATETIME` | **Drop** | Database bookkeeping |

In [2]:
KEEP = [
    "OGF_ID", "X", "Y",
    "FIRE_DISTURBANCE_AREA_IDENT", "FIRE_TYPE_CODE", "FIRE_YEAR",
    "FIRE_GENERAL_CAUSE_CODE", "FIRE_START_DATE", "FIRE_OUT_DATE", "FIRE_FINAL_SIZE",
]
DROP = {
    "FIRE_WEATHER_INDEX": "leakage: only recorded for fires",
    "FIRE_RESPONSE_OBJ_CODE": "leakage: decided after ignition",
    "OBJECTID": "duplicate of OGF_ID",
    "BUSINESS_EFFECTIVE_DATE": "database bookkeeping",
    "GEOMETRY_UPDATE_DATETIME": "database bookkeeping",
    "EFFECTIVE_DATETIME": "database bookkeeping",
    "SYSTEM_DATETIME": "database bookkeeping",
}

# Every raw column must be decided exactly once, so a new column in a future download is caught
decided = set(KEEP) | set(DROP)
assert set(KEEP).isdisjoint(DROP), "A column is in both KEEP and DROP"
assert set(raw.columns) <= decided, f"Undecided columns: {set(raw.columns) - decided}"
assert decided <= set(raw.columns), f"Expected columns missing from file: {decided - set(raw.columns)}"
print(f"All {raw.shape[1]} raw columns decided: {len(KEEP)} keep, {len(DROP)} drop")

All 17 raw columns decided: 10 keep, 7 drop


## Step 3. Remove the columns

In [3]:
step1 = raw[KEEP].copy()

before_mb = raw.memory_usage(deep=True).sum() / 1e6
after_mb = step1.memory_usage(deep=True).sum() / 1e6
print(f"Columns: {raw.shape[1]} -> {step1.shape[1]}")
print(f"Rows:    {len(raw):,} -> {len(step1):,}")
print(f"Memory:  {before_mb:.1f} MB -> {after_mb:.1f} MB")
step1.head()

Columns: 17 -> 10
Rows:    65,655 -> 65,655
Memory:  42.6 MB -> 22.3 MB


,OGF_ID,X,Y,FIRE_DISTURBANCE_AREA_IDENT,FIRE_TYPE_CODE,FIRE_YEAR,FIRE_GENERAL_CAUSE_CODE,FIRE_START_DATE,FIRE_OUT_DATE,FIRE_FINAL_SIZE
0,43129190,-88.0381,49.0228,NIP2,IFR,2003,RWY,2003/04/13 00:00:00+00,2003/04/14 00:00:00+00,1.5
1,43129191,-86.7287,49.8475,NIP20,IFR,2003,UNK,2003/05/27 00:00:00+00,2003/05/27 00:00:00+00,0.1
2,43129192,-88.0725,50.2296,NIP21,IFR,2003,RWY,2003/05/25 00:00:00+00,2003/05/25 00:00:00+00,15.0
3,43129193,-88.5994,53.4975,NIP22,IFR,2003,REC,2003/05/27 00:00:00+00,2003/05/28 00:00:00+00,0.4
4,43129194,-87.9607,50.2486,NIP23,IFR,2003,RWY,2003/05/25 00:00:00+00,2003/05/25 00:00:00+00,0.2


## Step 4. Check the result

Dropping columns must not change any row or any kept value.

In [4]:
checks = {
    "row count unchanged": len(step1) == len(raw),
    "OGF_ID still unique": step1["OGF_ID"].is_unique,
    "kept values identical to raw": step1.equals(raw[KEEP]),
    "no dropped column left": not (set(DROP) & set(step1.columns)),
    "no leakage column left": not ({"FIRE_WEATHER_INDEX", "FIRE_RESPONSE_OBJ_CODE"} & set(step1.columns)),
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values())

print("\nMissing values in the kept columns:")
display(step1.isna().sum().to_frame("missing").T)

PASS  row count unchanged
PASS  OGF_ID still unique
PASS  kept values identical to raw
PASS  no dropped column left
PASS  no leakage column left

Missing values in the kept columns:


,OGF_ID,X,Y,FIRE_DISTURBANCE_AREA_IDENT,FIRE_TYPE_CODE,FIRE_YEAR,FIRE_GENERAL_CAUSE_CODE,FIRE_START_DATE,FIRE_OUT_DATE,FIRE_FINAL_SIZE
missing,0,0,0,0,0,0,76,49,1202,0


The kept columns carry only three gaps, all known from the exploration notebook:

- **Cause** is missing only on prescribed burns, which Step 2 of cleaning will remove.
- **Start date** is missing on a few dozen fires. Those rows can't be labels and will be removed in Step 2 of cleaning.
- **Out date** is missing on about 1,200 fires. That only affects duration, so those rows stay.

## Step 5. Save

The output keeps the raw text formats. Dates are parsed in a later cleaning step, so this file stays an exact subset of the raw data.

In [5]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "fire_points_step1_columns.csv"
step1.to_csv(out_path, index=False)

reloaded = pd.read_csv(out_path, low_memory=False)
assert reloaded.shape == step1.shape and list(reloaded.columns) == KEEP
print(f"Saved {len(step1):,} rows x {step1.shape[1]} columns to {out_path.resolve()}")

Saved 65,655 rows x 10 columns to C:\Users\rishi\OneDrive\Desktop\Ai capstone\data\interim\fire_points_step1_columns.csv


## Next cleaning step

Step 2 removes rows that can't serve as ignition labels, using the rules from the exploration notebook:

- prescribed burns
- fires with no start date
- the 2003 season, whose dates are unreliable
- the 4 fires whose start year looks like a typo